
# Step 01: Merge raw publication data files

Input:
    data/raw/publication/*.csv

Output:
    data/interim/publication/publication_merged_long.csv

Purpose:
    - Concatenate publication-level data across years/journals
    - Apply minimal column standardization
    - Preserve all raw information


In [ ]:
import pandas as pd
from pathlib import Path
import pandas as pd
import os
from tqdm import tqdm
import gc
from joblib import Parallel, delayed
import xlrd
print(xlrd.__version__)

In [ ]:

# -------------------------------------------------
# 1. Define paths
# -------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parents[2]

RAW_PUB_DIR = PROJECT_ROOT / "data" / "raw" / "publication"
OUT_DIR = PROJECT_ROOT / "data" / "interim" / "publication"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "1_AllPublications.csv"


In [ ]:
# -------------------------------------------------
# 2. Functions
# -------------------------------------------------

def load_publication_files(input_dir: Path) -> list[pd.DataFrame]:
    """Load all CSV publication files from raw directory."""
    dfs = []
    for file in sorted(input_dir.glob("*.csv")):
        print(f"Loading {file.name}")
        df = pd.read_csv(file, low_memory=False)
        df["source_file"] = file.name
        dfs.append(df)
    return dfs


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names (minimal, non-destructive)."""
    df = df.rename(columns=lambda x: x.strip().lower())
    return df

In [ ]:
# -------------------------------------------------
# 3. Main merge logic
# -------------------------------------------------

def main():
    dfs = load_publication_files(RAW_PUB_DIR)
    print(f"Loaded {len(dfs)} files")

    dfs = [standardize_columns(df) for df in dfs]

    merged_df = pd.concat(dfs, axis=0, ignore_index=True)

    print("Merged dataset shape:", merged_df.shape)

    merged_df.reset_index(inplace=True, drop=False)
    merged_df.rename(columns={'index': 'index_num'}, inplace=True)

    merged_df.to_csv(OUT_FILE, index=False)
    print(f"Saved merged file to {OUT_FILE}")


if __name__ == "__main__":
    main()